# GLP-1 Drug Adverse Event Analytics & Machine Learning Pipeline

In [2]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
sns.set_style("whitegrid")

In [4]:
# All input files are expected here (adjust if your data lives elsewhere).
DATA_DIR = "/content"

# Where correlation-heatmap figures from the second half of the notebook are saved.
# Falls back to a local "outputs" folder when not running on Google Drive / Colab.
OUT = "/content/drive/MyDrive/outputs" if os.path.isdir("/content/drive/MyDrive") else "outputs"
os.makedirs(OUT, exist_ok=True)

## Google Colab setup (optional)
Only needed when running on Colab with data stored in Google Drive. Safe to skip (and safe to leave in) when running locally or on another platform.

In [5]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Load Data

In [9]:
ae_path = os.path.join(DATA_DIR, "adverse_events.csv")
df = pd.read_csv(ae_path, parse_dates=["receive_date"])
print(f"Loaded {len(df):,} rows, {df['safetyreportid'].nunique():,} unique reports")
# If this raises FileNotFoundError, double check DATA_DIR points at the folder
# containing adverse_events.csv (e.g. a mounted Google Drive path).

Loaded 149,209 rows, 54,170 unique reports


# 2. Cleaning & Feature Engineering

In [10]:
def age_to_years(age, unit):
    """Vectorized conversion of patient age to years given a unit column
    (values like 'Year', 'Month(s)', 'Week(s)', 'Day(s)'). Unknown/missing
    units are left as-is; missing ages stay NaN."""
    unit = unit.astype(str).str.lower()
    multiplier = pd.Series(1.0, index=age.index)
    multiplier[unit.str.startswith("month")] = 1 / 12
    multiplier[unit.str.startswith("week")] = 1 / 52
    multiplier[unit.str.startswith("day")] = 1 / 365
    years = age * multiplier
    years[age.isna()] = np.nan
    return years

df["age_years"] = age_to_years(df["patient_age"], df["patient_age_unit"])

# Report-year / month (useful for trend analysis)
df["report_year"] = df["receive_date"].dt.year
df["report_month"] = df["receive_date"].dt.month

# Number of distinct reactions reported per safety report (polypharmacy / severity proxy)
df["reactions_per_report"] = df.groupby("safetyreportid")["reaction"].transform("count")

# Target variable
TARGET = "serious"
n_missing_target = df[TARGET].isna().sum()
if n_missing_target:
    print(f"Dropping {n_missing_target:,} rows with missing '{TARGET}' label")
    df = df.dropna(subset=[TARGET])
df[TARGET] = df[TARGET].astype(bool)

# 3. Exploratory Data Analysis (EDA)

In [11]:
print("\n--- Class balance ---")
print(df[TARGET].value_counts(normalize=True).round(3))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Seriousness rate by drug
top_drugs = df["generic_name"].value_counts().head(10).index
sns.barplot(
    data=df[df["generic_name"].isin(top_drugs)],
    x="generic_name", y=TARGET, ax=axes[0, 0], errorbar=None,
)
axes[0, 0].set_title("Seriousness rate by drug (top 10 by volume)")
axes[0, 0].tick_params(axis="x", rotation=45)

# Age distribution by seriousness
sns.kdeplot(data=df, x="age_years", hue=TARGET, fill=True, ax=axes[0, 1], common_norm=False)
axes[0, 1].set_title("Age distribution by seriousness")

# Reports over time
trend = df.groupby(df["receive_date"].dt.to_period("Y")).size()
trend.index = trend.index.astype(str)
trend.plot(kind="line", marker="o", ax=axes[1, 0])
axes[1, 0].set_title("Adverse event reports per year")
axes[1, 0].tick_params(axis="x", rotation=45)

# Top reactions among serious cases
top_reactions = df[df[TARGET]]["reaction"].value_counts().head(10)
sns.barplot(x=top_reactions.values, y=top_reactions.index, ax=axes[1, 1])
axes[1, 1].set_title("Top 10 reactions in SERIOUS reports")

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150)
plt.close(fig)
print("Saved EDA figure -> eda_overview.png")


--- Class balance ---
serious
False    0.619
True     0.381
Name: proportion, dtype: float64
Saved EDA figure -> eda_overview.png


# 4. Modeling Setup

In [12]:
report_level = (
    df.groupby("safetyreportid")
    .agg(
        generic_name=("generic_name", "first"),
        country=("country", "first"),
        patient_sex=("patient_sex", "first"),
        age_years=("age_years", "first"),
        patient_weight_kg=("patient_weight_kg", "first"),
        report_year=("report_year", "first"),
        reactions_per_report=("reactions_per_report", "first"),
        n_unique_reactions=("reaction", "nunique"),
        serious=("serious", "first"),
    )
    .reset_index(drop=True)
)

num_features = ["age_years", "patient_weight_kg", "reactions_per_report",
                 "n_unique_reactions", "report_year"]
cat_features = ["generic_name", "country", "patient_sex"]

X = report_level[num_features + cat_features]
y = report_level[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), num_features),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_features),
    ]
)

# 5. Models

In [13]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    preds = pipe.predict(X_test)
    auc = roc_auc_score(y_test, proba)
    results[name] = {"pipeline": pipe, "auc": auc, "preds": preds, "proba": proba}
    print(f"\n=== {name} ===")
    print(f"ROC-AUC: {auc:.3f}")
    print(classification_report(y_test, preds, target_names=["Not serious", "Serious"]))

best_name = max(results, key=lambda k: results[k]["auc"])
best = results[best_name]
print(f"\nBest model: {best_name} (ROC-AUC = {best['auc']:.3f})")


=== LogisticRegression ===
ROC-AUC: 0.817
              precision    recall  f1-score   support

 Not serious       0.84      0.82      0.83      7734
     Serious       0.58      0.61      0.59      3100

    accuracy                           0.76     10834
   macro avg       0.71      0.72      0.71     10834
weighted avg       0.77      0.76      0.76     10834


=== RandomForest ===
ROC-AUC: 0.862
              precision    recall  f1-score   support

 Not serious       0.86      0.90      0.88      7734
     Serious       0.72      0.64      0.68      3100

    accuracy                           0.83     10834
   macro avg       0.79      0.77      0.78     10834
weighted avg       0.82      0.83      0.82     10834


=== GradientBoosting ===
ROC-AUC: 0.861
              precision    recall  f1-score   support

 Not serious       0.83      0.97      0.89      7734
     Serious       0.87      0.50      0.63      3100

    accuracy                           0.83     10834
   macr

# 6. Evaluation Plots for Best Model

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ConfusionMatrixDisplay.from_predictions(
    y_test, best["preds"], display_labels=["Not serious", "Serious"], ax=axes[0],
)
axes[0].set_title(f"Confusion Matrix - {best_name}")

fpr, tpr, _ = roc_curve(y_test, best["proba"])
axes[1].plot(fpr, tpr, label=f"AUC = {best['auc']:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

precision, recall, _ = precision_recall_curve(y_test, best["proba"])
axes[2].plot(recall, precision)
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.savefig("model_evaluation.png", dpi=150)
plt.close(fig)
print("Saved evaluation figure -> model_evaluation.png")

Saved evaluation figure -> model_evaluation.png


# 7. Feature Importance (tree-based models only)

In [15]:
if best_name in ("RandomForest", "GradientBoosting"):
    clf = best["pipeline"].named_steps["clf"]
    ohe = best["pipeline"].named_steps["prep"].named_transformers_["cat"].named_steps["onehot"]
    feature_names = num_features + list(ohe.get_feature_names_out(cat_features))
    importances = pd.Series(clf.feature_importances_, index=feature_names)
    top20 = importances.sort_values(ascending=False).head(20)

    fig = plt.figure(figsize=(8, 7))
    sns.barplot(x=top20.values, y=top20.index)
    plt.title(f"Top 20 Feature Importances - {best_name}")
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150)
    plt.close(fig)
    print("Saved feature importance figure -> feature_importance.png")
else:
    print(f"{best_name} has no feature_importances_ attribute; skipping this plot.")

Saved feature importance figure -> feature_importance.png


# 8. (Optional) Hyperparameter Tuning Example for Best Model Type
On the full ~54k-report dataset this grid search can take several minutes. Set `RUN_GRID_SEARCH = False` to skip it, or shrink `param_grid`.

In [16]:
RUN_GRID_SEARCH = False  # flip to True if you want to wait several minutes
if RUN_GRID_SEARCH and best_name == "RandomForest":
    param_grid = {
        "clf__n_estimators": [200, 400],
        "clf__max_depth": [8, 12, None],
        "clf__min_samples_leaf": [1, 5],
    }
    grid = GridSearchCV(
        Pipeline([("prep", preprocessor), ("clf", RandomForestClassifier(
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))]),
        param_grid, scoring="roc_auc", cv=3, n_jobs=-1,
    )
    grid.fit(X_train, y_train)
    print("\nBest params:", grid.best_params_)
    print("Best CV ROC-AUC:", round(grid.best_score_, 3))

print("\nDone. Outputs: eda_overview.png, model_evaluation.png, feature_importance.png")


Done. Outputs: eda_overview.png, model_evaluation.png, feature_importance.png


# 9. Additional Data Loading (Utilization & Adverse-Event Summary Data)

In [17]:
util = pd.read_csv(os.path.join(DATA_DIR, "cleaned_combined_data.csv"))
ae = pd.read_csv(ae_path)  # reloaded (unfiltered) for this section's own analysis
ae_summary = pd.read_csv(os.path.join(DATA_DIR, "adverse_events_summary.csv"))

# 10. Utilization / Cost Correlations (cleaned_combined_data.csv)

In [18]:
util_numeric_cols = [c for c in [
    "units_reimbursed", "total_amount_reimbursed", "medicaid_amount_reimbursed",
    "non_medicaid_amount_reimbursed", "reimbursement_per_prescription",
    "number_of_prescriptions", "units_per_prescription", "package_size",
] if c in util.columns]

util_corr = util[util_numeric_cols].corr()
util_corr_spearman = util[util_numeric_cols].corr(method="spearman")
print("=== Utilization/cost correlation (Pearson) ===\n", util_corr.round(3))
print("\n=== Utilization/cost correlation (Spearman) ===\n", util_corr_spearman.round(3))

fig = plt.figure()
sns.heatmap(util_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation: Medicaid Drug Utilization & Cost Fields")
plt.tight_layout()
plt.savefig(f"{OUT}/util_corr_heatmap.png", dpi=150)
plt.close(fig)

by_drug = util.groupby("molecule").agg(
    total_prescriptions=("number_of_prescriptions", "sum"),
    avg_cost_per_rx=("reimbursement_per_prescription", "mean"),
    total_reimbursed=("total_amount_reimbursed", "sum"),
).reset_index()
print("\nCorrelation across molecules (Rx volume vs avg cost/Rx):",
      round(by_drug[["total_prescriptions", "avg_cost_per_rx"]].corr().iloc[0, 1], 3))

=== Utilization/cost correlation (Pearson) ===
                                 units_reimbursed  total_amount_reimbursed  \
units_reimbursed                           1.000                    0.774   
total_amount_reimbursed                    0.774                    1.000   
medicaid_amount_reimbursed                 0.774                    1.000   
non_medicaid_amount_reimbursed             0.719                    0.911   
reimbursement_per_prescription             0.054                    0.051   
number_of_prescriptions                    0.315                    0.373   
units_per_prescription                     0.041                   -0.015   
package_size                              -0.022                    0.001   

                                medicaid_amount_reimbursed  \
units_reimbursed                                     0.774   
total_amount_reimbursed                              1.000   
medicaid_amount_reimbursed                           1.000   
non_medica

# 11. Adverse Event Summary Correlations (drug x reaction rows)

In [19]:
ae_summary_cols = [c for c in
    ["report_count", "pct_serious", "pct_hospitalization", "pct_death", "median_age", "pct_female"]
    if c in ae_summary.columns]
ae_corr = ae_summary[ae_summary_cols].corr()
print("\n=== Adverse event summary correlation (Pearson) ===\n", ae_corr.round(3))

fig = plt.figure()
sns.heatmap(ae_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation: Adverse Event Severity Metrics")
plt.tight_layout()
plt.savefig(f"{OUT}/ae_summary_corr_heatmap.png", dpi=150)
plt.close(fig)


=== Adverse event summary correlation (Pearson) ===
                      report_count  pct_serious  pct_hospitalization  \
report_count                1.000       -0.128               -0.076   
pct_serious                -0.128        1.000                0.556   
pct_hospitalization        -0.076        0.556                1.000   
pct_death                  -0.037        0.261                0.310   
median_age                  0.008        0.027                0.077   
pct_female                  0.013       -0.062               -0.094   

                     pct_death  median_age  pct_female  
report_count            -0.037       0.008       0.013  
pct_serious              0.261       0.027      -0.062  
pct_hospitalization      0.310       0.077      -0.094  
pct_death                1.000       0.160      -0.171  
median_age               0.160       1.000      -0.136  
pct_female              -0.171      -0.136       1.000  


# 12. Cross-Dataset: Utilization vs. Adverse Events, by Molecule

In [20]:
ae_reports = ae.drop_duplicates("safetyreportid")

ae_by_drug = ae_reports.groupby("generic_name").agg(
    total_reports=("safetyreportid", "nunique"),
    pct_serious=("serious", "mean"),
    pct_death=("seriousness_death", "mean"),
    pct_hospitalization=("seriousness_hospitalization", "mean"),
    median_age=("patient_age", "median"),
).reset_index()

util_by_drug = util.groupby("molecule").agg(
    total_prescriptions=("number_of_prescriptions", "sum"),
    total_units_reimbursed=("units_reimbursed", "sum"),
    total_amount_reimbursed=("total_amount_reimbursed", "sum"),
    avg_cost_per_rx=("reimbursement_per_prescription", "mean"),
).reset_index()

ae_by_drug["generic_name"] = ae_by_drug["generic_name"].str.lower().str.strip()
util_by_drug["molecule"] = util_by_drug["molecule"].str.lower().str.strip()

merged = pd.merge(util_by_drug, ae_by_drug, left_on="molecule", right_on="generic_name", how="inner")
print("\n=== Merged drug-level table ===\n", merged)
print("\nNote: util has combo molecules (e.g. 'insulin glargine + lixisenatide') "
      "that won't match ae's single-ingredient names — inner join drops them.")

cross_cols = [c for c in
    ["total_prescriptions", "total_units_reimbursed", "total_amount_reimbursed",
     "avg_cost_per_rx", "total_reports", "pct_serious", "pct_death", "pct_hospitalization"]
    if c in merged.columns]

if len(merged) >= 3:
    cross_corr = merged[cross_cols].corr(method="spearman")
    print("\n=== Cross-dataset correlation (Spearman, drug-level) ===\n", cross_corr.round(3))
    fig = plt.figure()
    sns.heatmap(cross_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
    plt.title("Correlation: Drug Utilization vs. Adverse Event Rates (by molecule)")
    plt.tight_layout()
    plt.savefig(f"{OUT}/cross_dataset_corr_heatmap.png", dpi=150)
    plt.close(fig)
else:
    print("\nToo few overlapping drug names for a reliable cross-dataset correlation.")


=== Merged drug-level table ===
        molecule  total_prescriptions  total_units_reimbursed  \
0   albiglutide             151885.0            7.882995e+05   
1   dulaglutide           24129880.0            1.054962e+08   
2     exenatide             731161.0            2.295801e+06   
3   liraglutide            9739833.0            1.597787e+08   
4  lixisenatide              87744.0            3.141200e+05   
5   semaglutide           32399294.0            3.147206e+08   
6   tirzepatide           10070719.0            4.089755e+07   

   total_amount_reimbursed  avg_cost_per_rx  generic_name  total_reports  \
0             1.042827e+08       723.202506   albiglutide           4982   
1             4.606194e+10       915.360066   dulaglutide           4994   
2             6.700684e+08       848.726650     exenatide           9236   
3             1.613229e+10       819.845597   liraglutide           9978   
4             4.807684e+07       815.990097  lixisenatide             19 

# 13. Significance Tests

In [21]:
def report_corr(data, col1, col2, method="spearman", label=""):
    """Print a correlation coefficient and p-value between two columns of `data`,
    dropping rows with missing values first. Skips (with a message) if fewer
    than 3 complete rows remain."""
    d = data[[col1, col2]].dropna()
    if len(d) < 3:
        print(f"{label}: not enough data")
        return
    corr_fn = spearmanr if method == "spearman" else pearsonr
    r, p = corr_fn(d[col1], d[col2])
    print(f"{label}: r={r:.3f}, p={p:.4f}, n={len(d)}")

print("\n=== Significance tests ===")
report_corr(util, "number_of_prescriptions", "reimbursement_per_prescription",
            label="Rx volume vs. cost/Rx")
report_corr(ae_summary, "report_count", "pct_serious", label="report_count vs. pct_serious")
report_corr(ae_summary, "median_age", "pct_death", label="median_age vs. pct_death")
if len(merged) >= 3:
    report_corr(merged, "total_prescriptions", "pct_serious",
                label="Rx volume vs. pct_serious (cross-dataset)")


=== Significance tests ===
Rx volume vs. cost/Rx: r=0.161, p=0.0000, n=77942
report_count vs. pct_serious: r=-0.263, p=0.0000, n=11093
median_age vs. pct_death: r=0.165, p=0.0000, n=8815
Rx volume vs. pct_serious (cross-dataset): r=0.429, p=0.3374, n=7
